# **Hugging Face Pipeline**
### 1. 개괄
- 주제: 언어 표현 변환기
- 내용: 사용자가 표현하고자 하는 메시지를 받고, 그 메시지를 공적인 언어, 정제된 언어 혹은 완곡한 표현으로 변환해주는 시스템

### 2. 모델
- 혐오 표현 분류: https://huggingface.co/hongssi/final_abuse_manual_model
- 문장 생성: https://huggingface.co/Qwen/Qwen2.5-3B-Instruct

In [ ]:
# !pip install --upgrade transformers

   ---------------------------------------- 0.0/10.7 MB ? eta -:--:--
   ----- ---------------------------------- 1.6/10.7 MB 9.4 MB/s eta 0:00:01
   -------------- ------------------------- 3.9/10.7 MB 10.2 MB/s eta 0:00:01
   ----------------------- ---------------- 6.3/10.7 MB 11.0 MB/s eta 0:00:01
   --------------------------------- ------ 8.9/10.7 MB 10.9 MB/s eta 0:00:01
   ---------------------------------------- 10.7/10.7 MB 11.1 MB/s  0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.2.0
    Uninstalling transformers-5.2.0:
      Successfully uninstalled transformers-5.2.0


In [ ]:
!pip install torch

##### 2-1. 모델 불러오기

In [ ]:
!pip install -U transformers accelerate safetensors sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 126.2 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForCausalLM,
)

In [ ]:
CLS_MODEL_ID = "hongssi/final_abuse_manual_model"
GEN_MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

LABELS = [
    "여성/가족", "남성", "성소수자", "인종/국적", "연령",
    "지역", "종교", "기타 혐오", "악플/욕설", "clean", "개인지칭"
]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] device: {DEVICE}")

[INFO] device: cuda


In [ ]:
# 혐오 표현 분류
cls_tokenizer = AutoTokenizer.from_pretrained(CLS_MODEL_ID)
cls_model = AutoModelForSequenceClassification.from_pretrained(CLS_MODEL_ID).to(DEVICE)
cls_model.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

ElectraForSequenceClassification(
  (electra): ElectraModel(
    (embeddings): ElectraEmbeddings(
      (word_embeddings): Embedding(30000, 768, padding_idx=3)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): ElectraEncoder(
      (layer): ModuleList(
        (0-11): 12 x ElectraLayer(
          (attention): ElectraAttention(
            (self): ElectraSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): ElectraSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): L

In [ ]:
# 문장 생성
gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_ID)
gen_model = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL_ID,
    torch_dtype="auto",
    device_map="auto"
)
gen_model.eval()

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-35): 36 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear(in_features=2048, out_features=256, bias=True)
          (v_proj): Linear(in_features=2048, out_features=256, bias=True)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (up_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((2048,), eps=1e-06)
    (ro

### 2-2. 분류 및 생성 함수

In [ ]:
# 혐오 표현 분류 함수 -> 모델 형태에 맞게.
@torch.no_grad()
def classify_text(text):
    inputs = cls_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    outputs = cls_model(**inputs)

    probs = torch.sigmoid(outputs.logits)[0].cpu()

    label_scores = {label: float(prob) for label, prob in zip(LABELS, probs)}

    harmful_scores = {k: v for k, v in label_scores.items() if k != "clean"}
    top3 = sorted(harmful_scores.items(), key=lambda x: x[1], reverse=True)[:3]
    top1_label, top1_score = top3[0]
    clean_score = label_scores["clean"]

    return label_scores, top3, top1_label, top1_score, clean_score

In [ ]:
# 문장 생성 함수
torch.no_grad()
def rewrite_text(text):
    messages = [
        {
            "role": "system",
            "content": (
                "너는 한국어 문장 순화 편집기다."
                "반드시 한국어로만 답하라. "
                "공격적, 모욕적, 배제적, 명령조 표현을 "
                "정중하고 중립적인 한국어 문장으로 고쳐라. "
                "입력 문장의 핵심 의도는 유지하되, "
                "욕설, 비하, 위협, 공격적 명령 표현은 직접적으로 반복하지 말고 "
                "완곡한 표현으로 바꿔라. "
                "설명하지 말고 고친 문장만 출력하라."
            )
        },
        {
            "role": "user",
            "content": (
                "예시 1\n"
                "입력: 너 같은 건 필요 없으니까 꺼져\n"
                "출력: 지금 당장 너와 함께 있는 건 무리일 것 같아.\n\n"
                "예시 2\n"
                "입력: 당신의 일처리는 쓰레기예요. 다시 해오세요.\n"
                "출력: 업무 처리에 미흡한 부분이 있어 다시 검토해 주시기 바랍니다.\n\n"
                f"입력: {text}\n"
                "출력:"
            )
        }
    ]

    prompt = gen_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = gen_tokenizer(prompt, return_tensors="pt").to(gen_model.device)

    output_ids = gen_model.generate(
        **inputs,
        max_new_tokens=70,
        do_sample=False,
        num_beams=5,
        no_repeat_ngram_size=3,
        repetition_penalty=1.2,
        length_penalty=1.1,
        early_stopping=True,
        pad_token_id=gen_tokenizer.eos_token_id
    )

    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    result = gen_tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    result = result.replace("출력:", "").strip()
    result = result.strip('"').strip("'").strip()

    return result

### 3. 실제 문장 순화

In [ ]:
user_text = input("문장을 입력하세요: ").strip()

if user_text:
    label_scores, top3, top1_label, top1_score, clean_score = classify_text(user_text)

    rewrite_needed = (top1_score >= 0.5) or (clean_score < 0.5)

    if rewrite_needed:
        rewritten = rewrite_text(user_text)
    else:
        rewritten = user_text

    print("\n[분류 결과]")
    print(f"clean 점수: {clean_score:.4f}")
    print(f"대표 유해 라벨: {top1_label} ({top1_score:.4f})")
    print("상위 3개 유해 라벨:")
    for label, score in top3:
        print(f"- {label}: {score:.4f}")

    print(f"\n순화 필요 여부: {rewrite_needed}")
    print(f"순화 결과: {rewritten}")

else:
    print("문장을 입력하지 않았습니다.")

문장을 입력하세요: 당신의 일처리는 정말 별로에요.

[분류 결과]
clean 점수: 0.0285
대표 유해 라벨: 악플/욕설 (0.9591)
상위 3개 유해 라벨:
- 악플/욕설: 0.9591
- 기타 혐오: 0.0092
- 개인지칭: 0.0080

순화 필요 여부: True
순화 결과: 당신의 일 처리에 대한 평가가 그렇게 좋지 않아요.
